# **Montagem do Google Drive, Preparação dos Dados (Limpeza e Estruturação) e Média Ponderada**

In [7]:
import pandas as pd
import os
from google.colab import drive
import numpy as np
import re # Necessário para a estruturação do Casa_2.txt

# ====================================================================
# 1. Montagem e Definição de Caminhos
# ====================================================================

print("Montando Google Drive...")
if not os.path.exists('/content/drive'):
    # Apenas monta se ainda não estiver montado
    drive.mount('/content/drive')
else:
    print("Drive já montado.")

PASTA_DRIVE = '/content/drive/MyDrive/Colab Notebooks/Projeto_dissertacao'

ARQUIVO_CASA_TXT = 'Casa_2.txt'
ARQUIVO_MATA_TXT = 'Mato_2.txt'

CAMINHO_SAVE_CASA = os.path.join(PASTA_DRIVE, 'Casa_2_estruturado.csv')
CAMINHO_SAVE_MATA = os.path.join(PASTA_DRIVE, 'Mato_2_estruturado.csv')

# ====================================================================
# 2. FUNÇÕES DE ESTRUTURAÇÃO ESPECÍFICAS (CORRIGIDAS)
# ====================================================================

def estruturar_mata(caminho_entrada, caminho_saida):
    """
    Estrutura o arquivo Mato_2.txt: [ID]\t[DATA]\t[HORA]\t[TEMP]\t[UMID]
    Lida com possíveis falhas de leitura do Timestamp.
    """
    print(f"\nIniciando estruturação de: {os.path.basename(caminho_entrada)}")

    try:
        # Lê o TXT. Assumindo separador de TAB (\t)
        df = pd.read_csv(caminho_entrada, sep='\t', header=None,
                         names=['Dispositivo', 'Data', 'Hora', 'Temperatura', 'Umidade'])
    except FileNotFoundError:
        print(f"ERRO: Arquivo não encontrado em {caminho_entrada}. Pulando.")
        return False
    except Exception as e:
        print(f"ERRO de leitura do Mato_2.txt. Verifique o separador ('\\t') e colunas: {e}")
        return False

    # 1. Combina Data e Hora
    if 'Data' not in df.columns or 'Hora' not in df.columns:
        print("ERRO CRÍTICO: As colunas 'Data' ou 'Hora' não foram lidas corretamente. Verifique o formato do arquivo Mato_2.txt.")
        return False

    df['Timestamp'] = df['Data'].astype(str) + ' ' + df['Hora'].astype(str)

    # 2. Conversão de Data/Hora (UserWarning sobre infer_datetime_format pode ser ignorada)
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce', infer_datetime_format=True)

    # 3. Limpeza e setar índice (Solução para o KeyError)
    # Remove linhas onde o Timestamp falhou (NaT) ANTES de setar o índice
    df.dropna(subset=['Timestamp', 'Dispositivo'], inplace=True)

    if df.empty:
        print("AVISO: O DataFrame Mata está vazio após a limpeza de Timestamp/Dispositivo.")
        return False

    df.set_index('Timestamp', inplace=True)

    # 4. Forçar tipos numéricos e limpeza final
    df['Temperatura'] = pd.to_numeric(df['Temperatura'], errors='coerce')
    df['Umidade'] = pd.to_numeric(df['Umidade'], errors='coerce')
    df.dropna(subset=['Temperatura', 'Umidade'], inplace=True)

    # 5. Salvar
    df[['Dispositivo', 'Temperatura', 'Umidade']].to_csv(caminho_saida)
    print(f"Estruturação de MATA concluída. Salvo em: {os.path.basename(caminho_saida)} (Total de {len(df)} linhas)")
    return True

def estruturar_casa(caminho_entrada, caminho_saida):
    """
    Estrutura o arquivo Casa_2.txt usando REGEX para extrair campos do log.
    """
    print(f"\nIniciando estruturação de: {os.path.basename(caminho_entrada)}")

    # Padrão REGEX para extrair os campos: Timestamp, Dispositivo, Temp e Umid
    padrao = re.compile(r"\[(?P<Timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\].*?Dispositivo: (?P<Dispositivo>[^,]+).*?Temp: (?P<Temp>[\d\.]+).*?Umid: (?P<Umid>[\d\.]+)")

    dados = []
    try:
        with open(caminho_entrada, 'r') as f:
            for linha in f:
                match = padrao.search(linha)
                if match:
                    dados.append(match.groupdict())
    except FileNotFoundError:
        print(f"ERRO: Arquivo não encontrado em {caminho_entrada}. Pulando.")
        return False

    df = pd.DataFrame(dados)
    if df.empty:
        print(f"ERRO: Nenhuma linha de dados válida encontrada em {caminho_entrada}.")
        return False

    # 1. Conversão de Data/Hora
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%Y-%m-%d %H:%M:%S', errors='coerce')

    # 2. Limpeza e setar índice
    df.dropna(subset=['Timestamp', 'Dispositivo'], inplace=True)
    df.set_index('Timestamp', inplace=True)

    # 3. Forçar tipos numéricos
    df['Temp'] = pd.to_numeric(df['Temp'], errors='coerce')
    df['Umid'] = pd.to_numeric(df['Umid'], errors='coerce')
    df.dropna(subset=['Temp', 'Umid'], inplace=True)

    # 4. Salvar
    df[['Dispositivo', 'Temp', 'Umid']].to_csv(caminho_saida)
    print(f"Estruturação de CASA concluída. Salvo em: {os.path.basename(caminho_saida)} (Total de {len(df)} linhas)")
    return True

# --- EXECUÇÃO DA ESTRUTURAÇÃO ---
estruturar_mata(os.path.join(PASTA_DRIVE, ARQUIVO_MATA_TXT), CAMINHO_SAVE_MATA)
estruturar_casa(os.path.join(PASTA_DRIVE, ARQUIVO_CASA_TXT), CAMINHO_SAVE_CASA)


# ====================================================================
# 3. Carregamento Reforçado (Usa os novos arquivos estruturados)
# ====================================================================

try:
    # Carregar df_mata com tratamento de erro de tipo
    df_mata = pd.read_csv(CAMINHO_SAVE_MATA, index_col=0, parse_dates=True)
    df_mata['Temperatura'] = pd.to_numeric(df_mata['Temperatura'], errors='coerce')
    df_mata['Umidade'] = pd.to_numeric(df_mata['Umidade'], errors='coerce')
    df_mata.dropna(subset=['Temperatura', 'Umidade'], inplace=True)

    # Carregar df_casa com tratamento de erro de tipo
    df_casa = pd.read_csv(CAMINHO_SAVE_CASA, index_col=0, parse_dates=True)
    df_casa['Temp'] = pd.to_numeric(df_casa['Temp'], errors='coerce')
    df_casa['Umid'] = pd.to_numeric(df_casa['Umid'], errors='coerce')
    df_casa.dropna(subset=['Temp', 'Umid'], inplace=True)

    print("\nDataFrames carregados e tipos de dados verificados com sucesso.")

except FileNotFoundError:
    print(f"\nERRO FATAL: Não foi possível encontrar os arquivos CSV estruturados. Verifique se os arquivos TXT foram criados corretamente.")
    exit()

# Verifica se os dataframes estão vazios após a limpeza
if df_mata.empty or df_casa.empty:
    print("\nERRO: Um dos DataFrames está vazio após o carregamento. Verifique se os arquivos TXT continham dados válidos.")
    exit()

# ====================================================================
# 4. Recálculo das Médias Ponderadas (Análise)
# ====================================================================

# A) Média Ponderada para Mata (df_mata)
# Esta seção agora deve rodar sem o AttributeError
duracao_mata = df_mata.index.to_series().diff().shift(-1).fillna(method='ffill')
if len(duracao_mata) > 1 and duracao_mata.iloc[-1].total_seconds() == 0:
    duracao_mata.iloc[-1] = duracao_mata.iloc[-2]
elif len(duracao_mata) == 1:
    duracao_mata.iloc[0] = pd.Timedelta(seconds=1)
pesos_mata = duracao_mata.dt.total_seconds()

media_temp_mata = np.average(df_mata['Temperatura'], weights=pesos_mata)
media_umid_mata = np.average(df_mata['Umidade'], weights=pesos_mata)


# B) Média Ponderada para Casa (df_casa), por Dispositivo
resultados_casa = []
grupos_dispositivo = df_casa.groupby('Dispositivo')

for dispositivo, df_grupo in grupos_dispositivo:
    if len(df_grupo) < 1:
        continue

    duracao_casa = df_grupo.index.to_series().diff().shift(-1).fillna(method='ffill')

    if len(duracao_casa) > 1 and duracao_casa.iloc[-1].total_seconds() == 0:
        duracao_casa.iloc[-1] = duracao_casa.iloc[-2]
    elif len(duracao_casa) == 1:
        duracao_casa.iloc[0] = pd.Timedelta(seconds=1)

    pesos_casa = duracao_casa.dt.total_seconds()

    media_temp_casa = np.average(df_grupo['Temp'], weights=pesos_casa)
    media_umid_casa = np.average(df_grupo['Umid'], weights=pesos_casa)

    resultados_casa.append({
        'Dispositivo': dispositivo,
        'Temp_Média_Ponderada': media_temp_casa,
        'Umid_Média_Ponderada': media_umid_casa
    })

df_resultados_casa = pd.DataFrame(resultados_casa).set_index('Dispositivo')


# ====================================================================
# 5. Comparação, Tabela e Conclusões
# ====================================================================

# Criar as colunas de comparação
df_resultados_casa['Diferença_Temp_vs_Mata (°C)'] = df_resultados_casa['Temp_Média_Ponderada'] - media_temp_mata
df_resultados_casa['Diferença_Umid_vs_Mata (%)'] = df_resultados_casa['Umid_Média_Ponderada'] - media_umid_mata

# Tabela final de comparação
df_comparacao = df_resultados_casa[[
    'Temp_Média_Ponderada',
    'Diferença_Temp_vs_Mata (°C)',
    'Umid_Média_Ponderada',
    'Diferença_Umid_vs_Mata (%)'
]].round(2)

print("\n" + "="*80)
print(f"|  Média Ponderada da MATA (Baseline - Mato_2): Temp: {media_temp_mata:.2f}°C | Umid: {media_umid_mata:.2f}%  |")
print("="*80 + "\n")

print("--- 📊 Tabela de Comparação Casa (Dispositivo - Casa_2) vs. Mata (Diferença) 📊 ---")
print(df_comparacao)

# -----------------
# Análise e Conclusões
# -----------------
conclusoes = []
temp_maior_que_mata = df_comparacao['Diferença_Temp_vs_Mata (°C)'].gt(0).sum()
umid_menor_que_mata = df_comparacao['Diferença_Umid_vs_Mata (%)'].lt(0).sum()
num_dispositivos = len(df_comparacao)

if num_dispositivos > 0:
    # Conclusão 1: Tendência de Temperatura
    if temp_maior_que_mata == num_dispositivos:
        conclusoes.append("→ A Casa é, em média, **mais quente** que a Mata em todas as áreas monitoradas.")
    elif temp_maior_que_mata == 0:
        conclusoes.append("→ A Casa é, em média, **mais fria** que a Mata em todas as áreas monitoradas.")
    else:
        conclusoes.append(f"→ Há uma variação: {temp_maior_que_mata}/{num_dispositivos} dispositivos na Casa são, em média, mais quentes que a Mata, indicando **microclimas internos** distintos.")

    # Conclusão 2: Tendência de Umidade
    if umid_menor_que_mata == num_dispositivos:
        conclusoes.append("→ A Casa é consistentemente **mais seca** que a Mata em todas as áreas.")
    elif umid_menor_que_mata == 0:
        conclusoes.append("→ A Casa é consistentemente **mais húmida** ou tem a mesma humidade que a Mata.")
    else:
        conclusoes.append(f"→ A maioria dos dispositivos ({umid_menor_que_mata}/{num_dispositivos}) indica uma Casa mais seca, mas alguns pontos podem estar com **humidade elevada**.")

    # Conclusão 3: Homogeneidade Interna
    std_temp = df_comparacao['Temp_Média_Ponderada'].std()
    conclusoes.append(f"→ **Disparidade Interna (Desvio Padrão Temp: {std_temp:.2f}°C):** Uma variação superior a 1 ou 2 graus indica grandes diferenças de temperatura entre os cômodos da Casa.")

print("\n--- 🧠 Conclusões da Análise Comparativa (Casa_2 vs. Mato_2) 🧠 ---")
for c in conclusoes:
    print(c)

Montando Google Drive...
Drive já montado.

Iniciando estruturação de: Mato_2.txt
Estruturação de MATA concluída. Salvo em: Mato_2_estruturado.csv (Total de 2048 linhas)

Iniciando estruturação de: Casa_2.txt
Estruturação de CASA concluída. Salvo em: Casa_2_estruturado.csv (Total de 312 linhas)

DataFrames carregados e tipos de dados verificados com sucesso.

|  Média Ponderada da MATA (Baseline - Mato_2): Temp: 25.76°C | Umid: 50.29%  |

--- 📊 Tabela de Comparação Casa (Dispositivo - Casa_2) vs. Mata (Diferença) 📊 ---
               Temp_Média_Ponderada  Diferença_Temp_vs_Mata (°C)  \
Dispositivo                                                        
Pico W_11f0bb                 32.26                         6.50   
Pico W_11f60f                 31.09                         5.32   

               Umid_Média_Ponderada  Diferença_Umid_vs_Mata (%)  
Dispositivo                                                      
Pico W_11f0bb                 42.61                       -7.68  
Pico

/tmp/ipython-input-3808184406.py:56: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce', infer_datetime_format=True)
/tmp/ipython-input-3808184406.py:159: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  duracao_mata = df_mata.index.to_series().diff().shift(-1).fillna(method='ffill')
/tmp/ipython-input-3808184406.py:178: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  duracao_casa = df_grupo.index.to_series().diff().shift(-1).fillna(method='ffill')
